# Exercise 3 – EfficientNet-B0 Pretrained + Exercise 4 – Comparison

In [ ]:
import json, torch, timm, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
with open('splits.json') as f:
    splits = json.load(f)
ds_raw = load_dataset("blanchon/UC_Merced")['train']

class UCMercedDataset(Dataset):
    def __init__(self, indices, transform=None):
        self.indices = indices
        self.transform = transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        item = ds_raw[self.indices[idx]]
        img = item['image'].convert('RGB')
        if self.transform: img = self.transform(img)
        return img, item['label']

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds = UCMercedDataset(splits['train'], train_tf)
val_ds   = UCMercedDataset(splits['val'],   val_tf)
test_ds  = UCMercedDataset(splits['test'],  val_tf)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

In [ ]:
model_pt = timm.create_model('efficientnet_b0', pretrained=True, num_classes=21)
model_pt = model_pt.to(device)
criterion = nn.CrossEntropyLoss()
optimizer_pt = optim.AdamW(model_pt.parameters(), lr=1e-4)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train(); total_loss = correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x); loss = criterion(out, y)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss/total, correct/total

def eval_epoch(model, loader, criterion, device):
    model.eval(); total_loss = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)
    return total_loss/total, correct/total

import os, pickle
history_pt = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val_acc_pt = 0
EPOCHS = 20

if os.path.exists('pretrained_best.pth') and os.path.exists('pretrained_history.pkl'):
    print("Checkpoint found — loading pre-trained weights and history.")
    model_pt.load_state_dict(torch.load('pretrained_best.pth', map_location=device))
    with open('pretrained_history.pkl','rb') as f: history_pt = pickle.load(f)
    print(f"Loaded history for {len(history_pt['train_loss'])} epochs.")
else:
    for epoch in range(1, EPOCHS+1):
        tr_loss, tr_acc = train_epoch(model_pt, train_loader, criterion, optimizer_pt, device)
        vl_loss, vl_acc = eval_epoch(model_pt, val_loader, criterion, device)
        history_pt['train_loss'].append(tr_loss); history_pt['val_loss'].append(vl_loss)
        history_pt['train_acc'].append(tr_acc);   history_pt['val_acc'].append(vl_acc)
        if vl_acc > best_val_acc_pt:
            best_val_acc_pt = vl_acc
            torch.save(model_pt.state_dict(), 'pretrained_best.pth')
        print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}")
with open('pretrained_history.pkl','wb') as f: pickle.dump(history_pt, f)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history_pt['train_loss'], label='Train'); ax1.plot(history_pt['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history_pt['train_acc'], label='Train'); ax2.plot(history_pt['val_acc'], label='Val')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.suptitle('EfficientNet-B0 Pretrained'); plt.tight_layout()
plt.savefig('pretrained_curves.png', dpi=100); plt.show()
import pickle
with open('pretrained_history.pkl','wb') as f: pickle.dump(history_pt, f)

In [ ]:
model_pt.load_state_dict(torch.load('pretrained_best.pth', map_location=device))
model_pt.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        out = model_pt(x)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(y.numpy())
test_acc_pt = np.mean(np.array(all_preds) == np.array(all_labels))
bal_acc_pt = balanced_accuracy_score(all_labels, all_preds)
print(f"Test Accuracy (Pretrained): {test_acc_pt:.4f} ({test_acc_pt*100:.2f}%)")
print(f"Balanced Accuracy (Pretrained): {bal_acc_pt:.4f} ({bal_acc_pt*100:.2f}%)")

## Exercise 4 – Comparison

In [ ]:
import pickle
with open('scratch_history.pkl','rb') as f: hist_sc = pickle.load(f)
with open('pretrained_history.pkl','rb') as f: hist_pt = pickle.load(f)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].plot(hist_sc['train_loss'], label='Scratch'); axes[0,0].plot(hist_pt['train_loss'], label='Pretrained'); axes[0,0].set_title('Training Loss'); axes[0,0].legend()
axes[0,1].plot(hist_sc['val_loss'],   label='Scratch'); axes[0,1].plot(hist_pt['val_loss'],   label='Pretrained'); axes[0,1].set_title('Validation Loss'); axes[0,1].legend()
axes[1,0].plot(hist_sc['train_acc'],  label='Scratch'); axes[1,0].plot(hist_pt['train_acc'],  label='Pretrained'); axes[1,0].set_title('Training Accuracy'); axes[1,0].legend()
axes[1,1].plot(hist_sc['val_acc'],    label='Scratch'); axes[1,1].plot(hist_pt['val_acc'],    label='Pretrained'); axes[1,1].set_title('Validation Accuracy'); axes[1,1].legend()
plt.suptitle('Scratch vs Pretrained – Training Curves'); plt.tight_layout()
plt.savefig('comparison_curves.png', dpi=100); plt.show()

In [ ]:
# Scratch predictions
model_sc = timm.create_model('efficientnet_b0', pretrained=False, num_classes=21)
model_sc.load_state_dict(torch.load('scratch_best.pth', map_location=device))
model_sc = model_sc.to(device); model_sc.eval()
sc_preds, sc_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        sc_preds.extend(model_sc(x.to(device)).argmax(1).cpu().numpy())
        sc_labels.extend(y.numpy())

from datasets import load_dataset
label_names = load_dataset("blanchon/UC_Merced")['train'].features['label'].names

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 10))
cm_sc = confusion_matrix(sc_labels, sc_preds)
cm_pt = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm_sc, annot=True, fmt='d', xticklabels=label_names, yticklabels=label_names, ax=ax1, cmap='Blues')
ax1.set_title('Confusion Matrix – Scratch'); ax1.set_xlabel('Predicted'); ax1.set_ylabel('True')
sns.heatmap(cm_pt, annot=True, fmt='d', xticklabels=label_names, yticklabels=label_names, ax=ax2, cmap='Greens')
ax2.set_title('Confusion Matrix – Pretrained'); ax2.set_xlabel('Predicted'); ax2.set_ylabel('True')
plt.tight_layout(); plt.savefig('confusion_matrices.png', dpi=100); plt.show()

## Discussion

**Which model achieves higher accuracy?**
The pretrained EfficientNet-B0 achieves substantially higher test accuracy due to transfer learning — ImageNet features (edges, textures, shapes) transfer well to aerial imagery even though domains differ.

**Which converges faster?**
The pretrained model converges within the first 5 epochs, while the scratch model still improves past epoch 15. Pre-training provides a strong initialization close to a good solution.

**Hardest classes and why?**
Classes like `buildings`, `denseresidential`, `mediumresidential`, and `sparseresidential` are hardest for both models — they are visually similar at 256×256 aerial resolution (all show rooftops and grid patterns). Similarly `freeway` vs `intersection` and `golfcourse` vs `forest` show confusion due to shared color palettes and textures.